# Step 6 Human Labeling Notebook

This notebook duplicates the Step 6 human-labeling logic in a self-contained Google Colab workflow. It reads Step 5's holdout file from Google Drive, reads the completed human-labeled holdout from the Step 6 Google Drive folder, creates consensus/agreement columns, computes reliability metrics, and writes Step 6 outputs back to Google Drive.

Run this notebook after Step 5 has created:

- `Team 8 - Capstone Project/Step 5 - Preprocess Split/data/holdout_1000.csv`

Step 6 also needs this labeled source file in Drive:

- `Team 8 - Capstone Project/Step 6 - Human Labeling/data/complete-labeled-holdout.csv`

Generated outputs are written to:

- `Team 8 - Capstone Project/Step 6 - Human Labeling/data/holdout_with_consensus.csv`
- `Team 8 - Capstone Project/Step 6 - Human Labeling/data/step6_reliability_report.csv`

## 1. Install Dependencies, Mount Drive, and Configure Paths

This section installs the reliability dependency, imports the libraries used by the Step 6 scripts, mounts Google Drive, and points inputs/outputs to the shared project folders.

In [ ]:
# Install reliability dependency in a fresh Colab runtime.
%pip install -q krippendorff scikit-learn pandas

from collections import Counter
from importlib import import_module
from pathlib import Path

import krippendorff
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# Mount Google Drive so Step 5 input and Step 6 outputs persist for everyone.
try:
    drive = import_module("google.colab").drive
except ModuleNotFoundError as exc:
    raise RuntimeError("Run this notebook in Google Colab so Google Drive can be mounted.") from exc

drive.mount("/content/drive")

# Make sure these exactly match the shared Drive folder structure.
PROJECT_DRIVE_BASE = Path("/content/drive/MyDrive/Team 8 - Capstone Project")
STEP5_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 5 - Preprocess Split"
STEP6_DRIVE_BASE = PROJECT_DRIVE_BASE / "Step 6 - Human Labeling"

STEP5_DATA_DIR = STEP5_DRIVE_BASE / "data"
DATA_DIR = STEP6_DRIVE_BASE / "data"

STEP5_HOLDOUT_FILE = STEP5_DATA_DIR / "holdout_1000.csv"
INPUT_FILE = DATA_DIR / "complete-labeled-holdout.csv"
CONSENSUS_FILE = DATA_DIR / "holdout_with_consensus.csv"
RELIABILITY_REPORT_FILE = DATA_DIR / "step6_reliability_report.csv"

SENTIMENT_COLUMNS = [
    "reviewer 1 sentiment",
    "reviewer 2 sentiment",
    "reviewer 3 sentiment",
]

LABELS = ["neither", "exploitation", "exploration", "ambiguous"]
LABEL_TO_CODE = {label: index for index, label in enumerate(LABELS)}
REVIEWER_PAIRS = [
    ("reviewer 1 sentiment", "reviewer 2 sentiment"),
    ("reviewer 1 sentiment", "reviewer 3 sentiment"),
    ("reviewer 2 sentiment", "reviewer 3 sentiment"),
]

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Step 5 holdout input: {STEP5_HOLDOUT_FILE}")
print(f"Step 6 labeled input: {INPUT_FILE}")
print(f"Step 6 data directory: {DATA_DIR}")

## 2. Verify Inputs

Step 6 uses Step 5's holdout as the source set that was labeled. The checks below confirm that the Step 5 holdout and the completed labeled holdout exist in Drive, and that the labeled file contains the required reviewer columns.

In [ ]:
def verify_drive_inputs():
    """Confirm required Drive inputs exist before running Step 6."""
    missing = []
    if not STEP5_HOLDOUT_FILE.exists():
        missing.append(
            f"Step 5 holdout not found: {STEP5_HOLDOUT_FILE}\n"
            "Run the Step 5 notebook first, or confirm the Step 5 Drive folder name is correct."
        )
    if not INPUT_FILE.exists():
        missing.append(
            f"Completed labeled holdout not found: {INPUT_FILE}\n"
            "Upload complete-labeled-holdout.csv to the Step 6 Drive data folder before running."
        )

    if missing:
        raise FileNotFoundError("\n\n".join(missing))

    labeled_preview = pd.read_csv(INPUT_FILE, dtype=str, encoding="utf-8-sig", nrows=5)
    required_columns = {"sentence_id", "sentence", *SENTIMENT_COLUMNS}
    missing_columns = sorted(required_columns - set(labeled_preview.columns))
    if missing_columns:
        raise ValueError(
            f"{INPUT_FILE} is missing required columns: {', '.join(missing_columns)}"
        )

    # Quick test to confirm the Step 6 Drive output folder is writable.
    test_file = DATA_DIR / "test_file.csv"
    pd.DataFrame({"check": [1, 2, 3]}).to_csv(test_file, index=False)

    print(f"Step 5 holdout exists: {STEP5_HOLDOUT_FILE}")
    print(f"Completed labeled holdout exists: {INPUT_FILE}")
    print(f"Step 6 Drive write test succeeded: {test_file}")


verify_drive_inputs()

## 3. Load and Compare Holdout Files

This section reads Step 5's holdout and the completed human-labeled holdout, then checks that the labeled rows align with the Step 5 holdout by `sentence_id`.

In [ ]:
def load_step6_inputs():
    """Load Step 5 holdout and completed Step 6 labels from Drive."""
    step5_holdout = pd.read_csv(STEP5_HOLDOUT_FILE, dtype=str, encoding="utf-8-sig")
    labeled = pd.read_csv(INPUT_FILE, dtype=str, encoding="utf-8-sig")

    if labeled["sentence_id"].duplicated().any():
        raise ValueError(f"{INPUT_FILE} contains duplicate sentence_id values")

    missing_from_labels = set(step5_holdout["sentence_id"]) - set(labeled["sentence_id"])
    extra_labels = set(labeled["sentence_id"]) - set(step5_holdout["sentence_id"])

    if missing_from_labels:
        print(f"Warning: {len(missing_from_labels)} Step 5 holdout rows are missing labels.")
    if extra_labels:
        print(f"Warning: {len(extra_labels)} labeled rows are not in the Step 5 holdout.")

    print(f"Step 5 holdout rows: {len(step5_holdout):,}")
    print(f"Completed labeled rows: {len(labeled):,}")

    return step5_holdout, labeled


step5_holdout, labeled_holdout = load_step6_inputs()
labeled_holdout.head(5)

## 4. Add Consensus and Agreement

This section duplicates `add_consensus.py`: each row gets a majority-vote `consensus` label and an `agreement` level of `unanimous`, `majority`, or `disagreement`.

In [ ]:
def get_consensus(row):
    votes = Counter(row[SENTIMENT_COLUMNS])
    label, count = votes.most_common(1)[0]

    if count >= 2:
        return label

    return "disagreement"


def get_agreement(row):
    votes = Counter(row[SENTIMENT_COLUMNS])
    highest_count = votes.most_common(1)[0][1]

    if highest_count == 3:
        return "unanimous"
    if highest_count == 2:
        return "majority"

    return "disagreement"


def add_consensus_columns(df):
    """Add consensus and agreement columns using the same logic as Step 6 script."""
    df = df.copy()
    df["consensus"] = df.apply(get_consensus, axis=1)
    df["agreement"] = df.apply(get_agreement, axis=1)
    return df


holdout_with_consensus = add_consensus_columns(labeled_holdout)

print("Consensus counts:")
print(holdout_with_consensus["consensus"].value_counts())

print("\nAgreement counts:")
print(holdout_with_consensus["agreement"].value_counts())

print("\nAgreement percentage breakdown:")
print(holdout_with_consensus["agreement"].value_counts(normalize=True).mul(100).round(2))

holdout_with_consensus.to_csv(CONSENSUS_FILE, index=False)
print(f"\nSaved output to {CONSENSUS_FILE}")

holdout_with_consensus.head(5)

## 5. Reliability Metrics

This section duplicates `step6_reliability.py`: it normalizes labels, validates class names, computes Krippendorff's alpha, computes pairwise Cohen's kappa, builds confusion matrices, and saves the reliability report.

In [ ]:
def normalize_label(label):
    if pd.isna(label):
        return pd.NA

    return str(label).strip().lower()


def build_reliability_matrix(df):
    return df[SENTIMENT_COLUMNS].T.to_numpy()


def build_alpha_matrix(reliability_matrix):
    coded_matrix = []
    for coder_labels in reliability_matrix:
        coded_matrix.append(
            [
                float("nan") if pd.isna(label) else LABEL_TO_CODE[label]
                for label in coder_labels
            ]
        )

    return coded_matrix


def validate_labels(df):
    observed_labels = set(df[SENTIMENT_COLUMNS].stack().dropna())
    unexpected_labels = observed_labels - set(LABELS)

    if unexpected_labels:
        raise ValueError(f"Unexpected sentiment labels found: {sorted(unexpected_labels)}")


def add_metric(report_rows, metric, value, pair="", label_a="", label_b=""):
    report_rows.append(
        {
            "metric": metric,
            "pair": pair,
            "label_a": label_a,
            "label_b": label_b,
            "value": value,
        }
    )


def compute_reliability_report(df):
    df = df.copy()
    df[SENTIMENT_COLUMNS] = df[SENTIMENT_COLUMNS].apply(lambda column: column.map(normalize_label))
    validate_labels(df)

    reliability_matrix = build_reliability_matrix(df)
    alpha_matrix = build_alpha_matrix(reliability_matrix)

    alpha = krippendorff.alpha(
        reliability_data=alpha_matrix,
        level_of_measurement="nominal",
    )

    report_rows = []
    add_metric(report_rows, "krippendorff_alpha_nominal", alpha)

    pairwise_kappas = []
    pairwise_matrices = {}

    for reviewer_a, reviewer_b in REVIEWER_PAIRS:
        pair_name = f"{reviewer_a} vs {reviewer_b}"
        pair_df = df[[reviewer_a, reviewer_b]].dropna()

        kappa = cohen_kappa_score(pair_df[reviewer_a], pair_df[reviewer_b], labels=LABELS)
        pairwise_kappas.append(kappa)
        add_metric(report_rows, "cohen_kappa", kappa, pair=pair_name)
        add_metric(report_rows, "pairwise_n_compared", len(pair_df), pair=pair_name)

        matrix = confusion_matrix(pair_df[reviewer_a], pair_df[reviewer_b], labels=LABELS)
        matrix_df = pd.DataFrame(matrix, index=LABELS, columns=LABELS)
        matrix_df.index.name = reviewer_a
        matrix_df.columns.name = reviewer_b
        pairwise_matrices[pair_name] = matrix_df

        for label_a in LABELS:
            for label_b in LABELS:
                add_metric(
                    report_rows,
                    "confusion_count",
                    int(matrix_df.loc[label_a, label_b]),
                    pair=pair_name,
                    label_a=label_a,
                    label_b=label_b,
                )

    average_kappa = sum(pairwise_kappas) / len(pairwise_kappas)
    add_metric(report_rows, "average_pairwise_cohen_kappa", average_kappa)

    report_df = pd.DataFrame(report_rows)
    return report_df, alpha, average_kappa, pairwise_matrices


reliability_report, alpha, average_kappa, pairwise_matrices = compute_reliability_report(holdout_with_consensus)
reliability_report.to_csv(RELIABILITY_REPORT_FILE, index=False)

print("Step 6 Reliability Summary")
print(f"Rows analyzed: {len(holdout_with_consensus)}")
print(f"Krippendorff's alpha (nominal): {alpha:.4f}")
print(f"Average pairwise Cohen's kappa: {average_kappa:.4f}")

print("\nPairwise Cohen's kappa:")
for _, row in reliability_report[reliability_report["metric"].eq("cohen_kappa")].iterrows():
    print(f"- {row['pair']}: {row['value']:.4f}")

print("\nPairwise confusion matrices:")
for pair_name, matrix_df in pairwise_matrices.items():
    print(f"\n{pair_name}")
    print(matrix_df)

print(f"\nSaved reliability report to {RELIABILITY_REPORT_FILE}")

## 6. Output Checks

These checks show the saved file paths, row counts, and first few rows so another person can confirm they reproduced the expected Step 6 results.

In [ ]:
saved_consensus = pd.read_csv(CONSENSUS_FILE, dtype=str, encoding="utf-8-sig")
saved_report = pd.read_csv(RELIABILITY_REPORT_FILE, encoding="utf-8-sig")

print("Generated files")
for output_file in [CONSENSUS_FILE, RELIABILITY_REPORT_FILE]:
    print(f"{output_file}: {output_file.exists()}")

print("\nGenerated row counts")
print(f"Holdout with consensus: {len(saved_consensus):,}")
print(f"Reliability report rows: {len(saved_report):,}")

saved_report.head(10)

## Notes on Reproducibility and Limitations

- This notebook duplicates the existing Step 6 scripts without changing them.
- The notebook reads Step 5's `holdout_1000.csv` directly from the Step 5 Google Drive `data/` folder to confirm the labeled file aligns with the holdout.
- The completed human-labeled file must already exist at `Step 6 - Human Labeling/data/complete-labeled-holdout.csv` before running this notebook.
- Reviewer labels are normalized to lowercase before reliability metrics are computed.
- Reliability metrics expect only these classes: `neither`, `exploitation`, `exploration`, and `ambiguous`.
- If rows are missing from the labeled file, the notebook warns about the mismatch; the reliability report still reflects the rows present in the completed labeled file.